# Round 1 bot-pattern research

Goal: treat the tape as bot-generated, extract repeatable structural order-book patterns, and turn them into passive market-making quote skews rather than spread-crossing entries.

In [33]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

candidates = [Path.cwd(), Path.cwd() / "datasets" / "round1"]
data_dir = next(path for path in candidates if (path / "prices_round_1_day_0.csv").exists())

PRODUCTS = ["ASH_COATED_OSMIUM", "INTARIAN_PEPPER_ROOT"]
DAYS = [-2, -1, 0]


def load_prices(day: int, product: str) -> pd.DataFrame:
    prices = pd.read_csv(data_dir / f"prices_round_1_day_{day}.csv", sep=";")
    prices = prices.loc[prices["product"] == product].sort_values("timestamp").copy()
    prices = prices.loc[
        prices["bid_price_1"].notna()
        & prices["ask_price_1"].notna()
        & prices["bid_price_2"].notna()
        & prices["ask_price_2"].notna()
    ].reset_index(drop=True)
    prices["spread"] = prices["ask_price_1"] - prices["bid_price_1"]
    prices["bid_gap"] = prices["bid_price_1"] - prices["bid_price_2"]
    prices["ask_gap"] = prices["ask_price_2"] - prices["ask_price_1"]
    prices["gap_skew"] = prices["ask_gap"] - prices["bid_gap"]
    prices["bid_levels"] = prices[["bid_price_1", "bid_price_2", "bid_price_3"]].notna().sum(axis=1)
    prices["ask_levels"] = prices[["ask_price_1", "ask_price_2", "ask_price_3"]].notna().sum(axis=1)
    prices["level_skew"] = prices["ask_levels"] - prices["bid_levels"]
    prices["alpha_1"] = prices["mid_price"].shift(-1) - prices["mid_price"]
    prices["alpha_5"] = prices["mid_price"].shift(-5) - prices["mid_price"]
    prices["alpha_10"] = prices["mid_price"].shift(-10) - prices["mid_price"]
    prices["alpha_20"] = prices["mid_price"].shift(-20) - prices["mid_price"]
    prices["alpha_30"] = prices["mid_price"].shift(-30) - prices["mid_price"]
    prices["alpha_40"] = prices["mid_price"].shift(-40) - prices["mid_price"]
    return prices


books = {(day, product): load_prices(day, product) for day in DAYS for product in PRODUCTS}
[(key, frame.shape) for key, frame in books.items()]

[((-2, 'ASH_COATED_OSMIUM'), (4251, 30)),
 ((-2, 'INTARIAN_PEPPER_ROOT'), (4221, 30)),
 ((-1, 'ASH_COATED_OSMIUM'), (4242, 30)),
 ((-1, 'INTARIAN_PEPPER_ROOT'), (4215, 30)),
 ((0, 'ASH_COATED_OSMIUM'), (4220, 30)),
 ((0, 'INTARIAN_PEPPER_ROOT'), (4170, 30))]

## Core hypothesis

The most useful passive signal is the ladder asymmetry between level 1 and level 2:

- `gap_skew = ask_gap - bid_gap`
- large positive `gap_skew`: the ask ladder is much wider than the bid ladder, which tends to precede upward mid-price drift
- large negative `gap_skew`: the bid ladder is much wider than the ask ladder, which tends to precede downward mid-price drift

That is exactly the kind of signal we can use to skew MM quotes: improve size and aggressiveness on the favored side, reduce or widen the other side.

In [34]:
def summarize_gap_rule(product: str) -> pd.DataFrame:
    rows = []
    for day in DAYS:
        df = books[(day, product)]
        bullish = df["ask_gap"] >= df["bid_gap"] + 3
        bearish = df["bid_gap"] >= df["ask_gap"] + 3
        neutral = ~(bullish | bearish)
        rows.extend(
            [
                {
                    "day": day,
                    "regime": "bullish_gap_rule",
                    "count": int(bullish.sum()),
                    "alpha_5": df.loc[bullish, "alpha_5"].mean(),
                    "alpha_10": df.loc[bullish, "alpha_10"].mean(),
                    "alpha_20": df.loc[bullish, "alpha_20"].mean(),
                    "alpha_30": df.loc[bullish, "alpha_30"].mean(),
                    "alpha_40": df.loc[bullish, "alpha_40"].mean()
                },
                {
                    "day": day,
                    "regime": "bearish_gap_rule",
                    "count": int(bearish.sum()),
                    "alpha_5": df.loc[bearish, "alpha_5"].mean(),
                    "alpha_10": df.loc[bearish, "alpha_10"].mean(),
                    "alpha_20": df.loc[bullish, "alpha_20"].mean(),
                    "alpha_30": df.loc[bullish, "alpha_30"].mean(),
                    "alpha_40": df.loc[bullish, "alpha_40"].mean()
                },
                {
                    "day": day,
                    "regime": "neutral",
                    "count": int(neutral.sum()),
                    "alpha_5": df.loc[neutral, "alpha_5"].mean(),
                    "alpha_10": df.loc[neutral, "alpha_10"].mean(),
                    "alpha_20": df.loc[bullish, "alpha_20"].mean(),
                    "alpha_30": df.loc[bullish, "alpha_30"].mean(),
                    "alpha_40": df.loc[bullish, "alpha_40"].mean()
                },
            ]
        )
    return pd.DataFrame(rows).round(3)


for product in PRODUCTS:
    print(f"\n=== {product} ===")
    display(summarize_gap_rule(product))


=== ASH_COATED_OSMIUM ===


,day,regime,count,alpha_5,alpha_10,alpha_20,alpha_30,alpha_40
0,-2,bullish_gap_rule,236,3.826,4.102,4.068,4.084,4.163
1,-2,bearish_gap_rule,249,-3.127,-3.306,4.068,4.084,4.163
2,-2,neutral,3766,-0.043,-0.065,4.068,4.084,4.163
3,-1,bullish_gap_rule,237,3.958,4.040,3.968,3.932,4.071
4,-1,bearish_gap_rule,237,-3.247,-3.303,3.968,3.932,4.071
5,-1,neutral,3768,-0.033,-0.022,3.968,3.932,4.071
6,0,bullish_gap_rule,244,4.154,3.895,4.002,3.913,3.775
7,0,bearish_gap_rule,235,-3.436,-3.228,4.002,3.913,3.775
8,0,neutral,3741,-0.047,-0.034,4.002,3.913,3.775



=== INTARIAN_PEPPER_ROOT ===


,day,regime,count,alpha_5,alpha_10,alpha_20,alpha_30,alpha_40
0,-2,bullish_gap_rule,112,5.562,6.933,9.152,11.638,13.884
1,-2,bearish_gap_rule,108,-2.935,-1.694,9.152,11.638,13.884
2,-2,neutral,4001,1.173,2.350,9.152,11.638,13.884
3,-1,bullish_gap_rule,120,5.950,6.871,9.346,11.671,14.129
4,-1,bearish_gap_rule,123,-3.317,-2.004,9.346,11.671,14.129
5,-1,neutral,3972,1.181,2.370,9.346,11.671,14.129
6,0,bullish_gap_rule,111,6.230,7.653,9.833,12.459,14.905
7,0,bearish_gap_rule,106,-3.533,-2.311,9.833,12.459,14.905
8,0,neutral,3953,1.184,2.375,9.833,12.459,14.905


In [35]:
def exact_state_table(product: str, day: int = 0, min_count: int = 20) -> pd.DataFrame:
    df = books[(day, product)]
    states = (
        df.groupby(["spread", "bid_gap", "ask_gap"])
        .agg(
            count=("mid_price", "size"),
            alpha_1=("alpha_1", "mean"),
            alpha_5=("alpha_5", "mean"),
            alpha_10=("alpha_10", "mean"),
        )
        .reset_index()
    )
    states = states.loc[states["count"] >= min_count].sort_values(["alpha_5", "count"], ascending=[False, False])
    return states.round(3)


for product in PRODUCTS:
    print(f"\n=== day 0 exact states: {product} ===")
    display(exact_state_table(product))


=== day 0 exact states: ASH_COATED_OSMIUM ===


,spread,bid_gap,ask_gap,count,alpha_1,alpha_5,alpha_10
4,6.0,2.0,10.0,44,5.239,5.239,4.864
1,5.0,3.0,11.0,42,5.607,5.226,5.714
14,9.0,3.0,7.0,58,3.224,3.621,3.167
16,10.0,2.0,6.0,53,3.302,3.330,2.953
24,16.0,2.0,3.0,1884,0.153,0.186,0.242
26,16.0,3.0,3.0,20,-0.075,0.050,-0.775
25,16.0,3.0,2.0,1799,-0.262,-0.303,-0.322
20,11.0,5.0,2.0,67,-2.896,-2.761,-2.627
18,10.0,6.0,3.0,74,-2.682,-2.858,-2.730
10,7.0,9.0,2.0,23,-4.652,-4.696,-4.587



=== day 0 exact states: INTARIAN_PEPPER_ROOT ===


,spread,bid_gap,ask_gap,count,alpha_1,alpha_5,alpha_10
10,3.0,3.0,11.0,35,5.757,6.686,8.400
0,2.0,3.0,11.0,25,5.760,6.400,7.620
46,14.0,3.0,4.0,107,0.561,1.652,2.779
42,13.0,3.0,4.0,279,0.565,1.303,2.663
41,13.0,3.0,3.0,1273,0.191,1.209,2.359
45,14.0,3.0,3.0,1731,0.176,1.175,2.334
43,13.0,4.0,3.0,264,0.119,1.080,2.316
44,13.0,4.0,4.0,127,0.465,1.016,2.356
47,14.0,4.0,3.0,102,0.098,0.718,2.178
12,3.0,10.0,3.0,24,-5.021,-3.875,-2.396


In [36]:
def dominant_templates(product: str, day: int = 0, min_count: int = 20) -> pd.DataFrame:
    df = books[(day, product)]
    templates = (
        df.groupby(["spread", "bid_gap", "ask_gap", "level_skew"])
        .agg(count=("mid_price", "size"), alpha_5=("alpha_5", "mean"))
        .reset_index()
        .loc[lambda x: x["count"] >= min_count]
        .sort_values(["count", "alpha_5"], ascending=[False, False])
    )
    return templates.round(3)


for product in PRODUCTS:
    print(f"\n=== day 0 dominant templates: {product} ===")
    display(dominant_templates(product).head(12))


=== day 0 dominant templates: ASH_COATED_OSMIUM ===


,spread,bid_gap,ask_gap,level_skew,count,alpha_5
32,16.0,2.0,3.0,0,1884,0.186
33,16.0,3.0,2.0,0,1799,-0.303
27,11.0,5.0,2.0,-1,62,-2.718
24,10.0,6.0,3.0,-1,59,-2.864
19,9.0,3.0,7.0,1,47,3.777
2,5.0,3.0,11.0,1,38,5.145
22,10.0,2.0,6.0,1,38,3.105
6,6.0,2.0,10.0,1,37,5.203
9,6.0,10.0,3.0,-1,23,-4.630
34,16.0,3.0,3.0,0,20,0.050



=== day 0 dominant templates: INTARIAN_PEPPER_ROOT ===


,spread,bid_gap,ask_gap,level_skew,count,alpha_5
62,14.0,3.0,3.0,0,1731,1.175
58,13.0,3.0,3.0,0,1273,1.209
59,13.0,3.0,4.0,0,279,1.303
60,13.0,4.0,3.0,0,264,1.080
61,13.0,4.0,4.0,0,127,1.016
63,14.0,3.0,4.0,0,107,1.652
64,14.0,4.0,3.0,0,102,0.718
16,3.0,3.0,11.0,1,31,6.710
31,4.0,10.0,3.0,-1,30,-3.983
18,3.0,10.0,3.0,-1,20,-3.900


## Practical MM translation

Use the signal as a quote skew, not as a taker trigger.

For `bullish_gap_rule` (`ask_gap >= bid_gap + 3`):

- improve the bid by 1 tick if the spread allows it
- increase bid size
- keep the ask wider or smaller

For `bearish_gap_rule` (`bid_gap >= ask_gap + 3`):

- improve the ask by 1 tick if the spread allows it
- increase ask size
- keep the bid wider or smaller

Day-0 exact templates worth encoding first:

- `ASH_COATED_OSMIUM`: bullish `(spread, bid_gap, ask_gap)` of `(5,3,11)`, `(6,2,10)`, `(9,3,7)`, `(10,2,6)`; bearish mirrors `(7,9,2)`, `(6,10,3)`, `(11,5,2)`, `(10,6,3)`
- `INTARIAN_PEPPER_ROOT`: strongest bullish states `(2,3,11)` and `(3,3,11)`; strongest bearish states `(4,10,3)` and `(3,10,3)`

The broad gap rule is more robust across days, while the exact templates are stronger but more selective.